## Domain Model Test

In [ ]:
# %pip install yfinance

In [ ]:
from domain.usd_market_data import USDMarketData

market_data = USDMarketData(
    stock_symbol="NVDA",
    price=180.23,
    volume=80_000_000,
)

## Adapter Interface Test

In [ ]:
from domain.usd_market_data_provider import USDMarketDataProvider
from domain.usd_market_data import USDMarketData


class FakeMarketDataProvider(USDMarketDataProvider):

    def get_market_data(self, stock_symbol: str) -> USDMarketData:
        return USDMarketData(
            stock_symbol=stock_symbol.upper(),
            price=180.25,
            volume=50_000_000,
        )


provider: USDMarketDataProvider = FakeMarketDataProvider()

data = provider.get_market_data("nvda")

data


## Live Yahoo Test

In [ ]:
from tools.usd_yahoo_finance_adapter import USDYahooFinanceAdapter

provider = USDYahooFinanceAdapter()

data = provider.get_market_data("NVDA")

data

## Test Analyzer

1. Input domain model
2. MarketAgent.Analyze()
3. Output Domain Model
4. Analysis Result

In [ ]:
from tools.usd_yahoo_finance_adapter import USDYahooFinanceAdapter
from agents.usd_market_agent import USDMarketAgent

provider = USDYahooFinanceAdapter()
agent = USDMarketAgent()

market_data = provider.get_market_data("NVDA")

print(market_data)

result = agent.analyze(market_data)

print(result)

# Register the Market Agent

"NVDA" -> ResearchPlanner -> ResearchTask(task_type="market_analysis", stock_symbol="NVDA")->AgentRouter->AgentRegistry->MarketAgent

In [ ]:
from orchestration.usd_agent_registry import USDAgentRegistry
from orchestration.usd_agent_router import USDAgentRouter
from agents.usd_market_agent import USDMarketAgent
from orchestration.usd_research_planner import USDResearchPlanner

# 1. Create registry
registry = USDAgentRegistry()

# 2. Create specialist
market_agent = USDMarketAgent()

# 3. Register specialist
registry.register(
    "market_analysis",
    market_agent
)

# 4. Create router
router = USDAgentRouter(registry)

# 5. Ask router for appropriate agent
agent = router.route("market_analysis")

# agent

planner = USDResearchPlanner()

tasks = planner.plan("NVDA")

task = tasks[0]

# Taking first task
print(task)
print(agent)
print(type(agent))

## Connect Provider Data

Planner driven pipeline
"NVDA" -> ResearchPlanner -> ResearchTask(task_type="market_analysis", stock_symbol="NVDA")->AgentRouter->AgentRegistry->MarketAgent->MarketData->YahooFinanceAdapter->AnalysisResult


In [ ]:
from tools.usd_yahoo_finance_adapter import USDYahooFinanceAdapter

provider = USDYahooFinanceAdapter()

# Planner creates tasks
tasks = planner.plan("NVDA")

# Select market task
task = next(
    task
    for task in tasks
    if task.task_type == "market_analysis"
)

# Router selects appropriate specialist
agent = router.route(task.task_type)

# Adapter retrieves normalized market data
market_data = provider.get_market_data(task.stock_symbol)

# Specialist analyzes data
result = agent.analyze(market_data)

print("TASK")
print(task)

print("\nAGENT")
print(type(agent).__name__)

print("\nMARKET DATA")
print(market_data)

print("\nANALYSIS RESULT")
print(result)

## Research Planner

Now the router consumes the task

In [ ]:
from orchestration.usd_research_planner import USDResearchPlanner

planner = USDResearchPlanner()

tasks = planner.plan("NVDA")

print(tasks)

## Connect the Planner to Router


In [ ]:
from agents.usd_market_agent import USDMarketAgent
from orchestration.usd_agent_registry import USDAgentRegistry
from orchestration.usd_agent_router import USDAgentRouter
from orchestration.usd_research_planner import USDResearchPlanner

researchPlanner = USDResearchPlanner()

registry = USDAgentRegistry()

registry.register(
    "market_analysis",
    USDMarketAgent()
)

router = USDAgentRouter(registry)

agent = router.route("market_analysis")

print(agent)
print(type(agent))


<class 'agents.usd_market_agent.USDMarketAgent'>


## The code will explicitly choose the specialist

NVDA

YahooFinanceAdapter
 
Yahoo Finance
 
Normalization
 
MarketData
 
"market_analysis"
 
AgentRouter
 
AgentRegistry
 
MarketAgent
 
analyze(MarketData)
 
AnalysisResult

In [8]:
from agents.usd_market_agent import USDMarketAgent
from orchestration.usd_agent_registry import USDAgentRegistry
from orchestration.usd_agent_router import USDAgentRouter
from orchestration.usd_research_planner import USDResearchPlanner
from tools.usd_yahoo_finance_adapter import USDYahooFinanceAdapter

provider = USDYahooFinanceAdapter()

registry = USDAgentRegistry()

registry.register(
    "market_analysis",
    USDMarketAgent()
)

router = USDAgentRouter(registry)

# Get Normalized data
market_data = provider.get_market_data("NVDA")

# Route the Task
agent = router.route("market_analysis")

# Run the Strategy
result = agent.analyze(market_data)

print(result)
print(result.agent)
print(result.summary)
print(result.evidence)

USDAnalysisResult(agent='market', evidence=['volume=101482897price=239.24'], summary='NVDA is trading at value: $239.24.')
market
NVDA is trading at value: $239.24.
['volume=101482897price=239.24']
